# 🌾 KrishiRakshak — 38-Class Full Model Training (Google Colab T4 GPU)

This notebook trains the **MobileNetV2** model on the **full 38-class PlantVillage dataset (all 14 crops)** using PyTorch on a free Colab GPU, exports the model to ONNX, and downloads `best_model.pt` directly to your computer.

### ⚡ Step 0: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU is not active. Please set Runtime > Change runtime type > T4 GPU")

In [ ]:
# 2. Clone the KrishiRakshak repository
!git clone https://github.com/AkshithCodez/KrishiRakshak.git
%cd KrishiRakshak
!git pull origin main

In [ ]:
# 3. Download the FULL 38-Class PlantVillage Dataset
import os
import kagglehub

!pip install -q kagglehub onnx onnxscript

print("Downloading the full 38-class PlantVillage dataset from Kaggle...")
download_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print(f"Raw dataset downloaded to: {download_path}")

# Locate the directory containing all 38 class folders
train_folder = None
for root, dirs, files in os.walk(download_path):
    if len(dirs) == 38:
        train_folder = root
        break
    elif "train" in dirs and len(os.listdir(os.path.join(root, "train"))) == 38:
        train_folder = os.path.join(root, "train")
        break

if not train_folder:
    train_folder = download_path

classes = [d for d in os.listdir(train_folder) if os.path.isdir(os.path.join(train_folder, d))]
print("\n" + "="*50)
print(f"✓ Verified Dataset Path: {train_folder}")
print(f"✓ Total Classes Found: {len(classes)} / 38")
print("="*50)

In [ ]:
# 4. Train MobileNetV2 on all 38 classes (~18-25 mins on Colab T4 GPU)
!python ml/src/train.py \
    --data-dir "$train_folder" \
    --epochs 15 \
    --batch-size 32 \
    --head-lr 0.001 \
    --output-dir ml/models

In [ ]:
# 5. Export trained 38-class model to ONNX format
!python ml/src/export_onnx.py \
    --model-path ml/models/best_model.pt \
    --output ml/models/model.onnx

In [ ]:
# 6. Download the 38-class model files directly to your PC
from google.colab import files

print("Downloading 38-class best_model.pt...")
files.download('ml/models/best_model.pt')

if os.path.exists('ml/models/model.onnx'):
    print("Downloading model.onnx...")
    files.download('ml/models/model.onnx')

if os.path.exists('ml/models/class_info.json'):
    print("Downloading class_info.json...")
    files.download('ml/models/class_info.json')